# Image, Bounding Box, and IoU

Goal: understand how an object detector represents object location and how IoU measures localization quality.

We start with a synthetic image so the first lesson does not depend on any external dataset.

## Motivation

In object detection, the model must answer two questions:

1. What object is present?
2. Where is it?

A bounding box is one common way to represent the second answer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

image = np.full((320, 480, 3), 245, dtype=np.uint8)

# Box format: x_min, y_min, x_max, y_max
ground_truth = (120, 80, 330, 240)
prediction = (150, 105, 360, 250)

def draw_box(ax, box, color, label):
    x_min, y_min, x_max, y_max = box
    width = x_max - x_min
    height = y_max - y_min
    ax.add_patch(Rectangle((x_min, y_min), width, height, fill=False, edgecolor=color, linewidth=2))
    ax.text(x_min, y_min - 8, label, color=color, fontsize=11)

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(image)
draw_box(ax, ground_truth, "tab:green", "ground truth")
draw_box(ax, prediction, "tab:red", "prediction")
ax.set_axis_off()
plt.show()

## Formal Definition

Intersection over Union is:

```text
IoU = area(intersection) / area(union)
```

It is high when two boxes overlap strongly and low when they barely overlap.

In [ ]:
def box_area(box):
    x_min, y_min, x_max, y_max = box
    return max(0, x_max - x_min) * max(0, y_max - y_min)

def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = box_area((ix1, iy1, ix2, iy2))
    union = box_area(box_a) + box_area(box_b) - intersection
    return intersection / union if union > 0 else 0.0

iou_value = iou(ground_truth, prediction)
iou_value

## Critical Thinking

If this prediction has correct class but IoU is below the evaluation threshold, should we call it a useful detection or a failure?

Think in terms of the real application. For example, helmet detection may tolerate a slightly imperfect box, while defect detection may require much tighter localization.

## Assignment

Change the predicted box and observe how IoU changes.

Answer these questions:

1. Which movement hurts IoU more: shifting the box or resizing it?
2. Can a box look visually acceptable but still fail at IoU 0.75?
3. What does this imply for annotation quality?